# 01 — LOB Data Exploration

Walks through a single LOBSTER sample day:
1. Load message and orderbook files
2. Inspect event types and timing
3. Visualize the book and mid-price
4. Quick stats: spread distribution, depth, trade rate

**Prerequisite:** download a LOBSTER sample into `../data/raw/`. See `data/README.md`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.lobster import find_lobster_pair, load_paired, compute_mid_price, compute_microprice, compute_spread
from src.data.features import build_feature_panel, make_targets

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Locate and load the LOBSTER sample

In [ ]:
files = find_lobster_pair('../data/raw', ticker='AAPL')
print(f'Ticker: {files.ticker}')
print(f'Date: {files.date}')
print(f'Levels: {files.levels}')
print(f'Time range: {files.start_time}s .. {files.end_time}s')

messages, orderbook = load_paired(files)
print(f'Loaded {len(messages):,} events')

## 2. Event-type distribution

What fraction of events are submissions vs cancellations vs executions?

In [ ]:
event_counts = messages['type_name'].value_counts()
print(event_counts)

event_counts.plot.bar(color='steelblue')
plt.title('Event type distribution')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Mid-price evolution through the day

In [ ]:
mid = compute_mid_price(orderbook)
micro = compute_microprice(orderbook)

# Convert seconds-after-midnight to hours for the x-axis
time_hours = messages['time'] / 3600.0

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_hours, mid, label='mid', linewidth=0.7, color='steelblue')
ax.plot(time_hours, micro, label='microprice', linewidth=0.7, color='crimson', alpha=0.5)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Price ($)')
ax.set_title(f'{files.ticker} mid-price evolution, {files.date}')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Spread statistics

In [ ]:
spread = compute_spread(orderbook)
print(f'Spread stats ($):')
print(f'  median: {spread.median():.4f}')
print(f'  mean:   {spread.mean():.4f}')
print(f'  90th:   {spread.quantile(0.9):.4f}')
print(f'  99th:   {spread.quantile(0.99):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(spread * 100, bins=60, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Spread (cents)')
axes[0].set_ylabel('Count')
axes[0].set_title('Spread distribution')
axes[0].axvline(spread.median() * 100, color='red', linestyle='--', label='median')
axes[0].legend()

axes[1].plot(time_hours, spread * 100, linewidth=0.4, color='steelblue')
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Spread (cents)')
axes[1].set_title('Spread through the day')
plt.tight_layout()
plt.show()

## 5. Build the feature panel

In [ ]:
features = build_feature_panel(messages, orderbook, ofi_max_level=files.levels)
print(f'Feature panel: {features.shape}')
features.head(10)

In [ ]:
features.describe().T

## 6. Targets at multiple horizons

Class balance: how often does the mid actually move at each horizon?

In [ ]:
targets = make_targets(orderbook, horizons=(1, 5, 30))

for h in (1, 5, 30):
    counts = targets[f'target_dir_h{h}'].value_counts().sort_index()
    total = counts.sum()
    print(f'Horizon h={h}:')
    for cls in (-1, 0, 1):
        if cls in counts.index:
            print(f'  {cls:+d}: {counts[cls]:>8,} ({counts[cls]/total*100:.1f}%)')
    print()

## 7. OFI vs future mid return

A quick visual sanity check: does OFI at the level-1 book correlate with the next-bar return?

In [ ]:
ofi_L1 = features['ofi_L1']
future_ret = np.log(mid.shift(-5) / mid)

valid = (~ofi_L1.isna()) & (~future_ret.isna())
corr = np.corrcoef(ofi_L1[valid], future_ret[valid])[0, 1]
print(f'Correlation(OFI_L1, future 5-event return): {corr:.4f}')

# Bin OFI into deciles and look at mean future return per bin
df = pd.DataFrame({'ofi': ofi_L1[valid], 'ret': future_ret[valid]})
df['ofi_decile'] = pd.qcut(df['ofi'], 10, labels=False, duplicates='drop')
decile_means = df.groupby('ofi_decile')['ret'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(decile_means.index, decile_means.values * 1e4, color='steelblue')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('OFI decile (low to high)')
ax.set_ylabel('Mean future 5-event return (bps)')
ax.set_title('Future return by OFI decile — a strong monotonic pattern = signal')
plt.tight_layout()
plt.show()

## Next steps

→ `02_baseline_models.ipynb`: Run the persistence / linear / XGBoost baselines using walk-forward CV.

Or from the command line:
```bash
python -m src.training.train_baselines --ticker AAPL --target-horizon 5
```